# Projeto Retinopatia

Classificação binária de retinopatia diabética em fotos de fundo de olho,
usando transfer learning com EfficientNet.

Este notebook é apenas o **orquestrador da execução no Colab**: toda a lógica
vive nos módulos Python do repositório (`src/retinopatia/`), que são editados
no VS Code e trazidos para cá pelo `git clone`/`git pull` da primeira célula.

## Preparação do Repositório no Colab

In [ ]:
# O código do projeto vive no repositório do GitHub.
# Esta célula traz (ou atualiza) esse código dentro do Colab.

import os
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"

# Versão do código a usar. "origin/main" é a mais recente no GitHub.
# Para reproduzir um treino antigo, use a tag ou o commit dele
# (por exemplo "treino-v2"); o commit de cada treino fica no
# configuracao.json da pasta de resultados e no registro de experimentos.
REVISAO_CODIGO = "origin/main"

# Rede usada neste treino: "EfficientNetB4" ou "EfficientNetB0"
# (outras opções em configuracao.py). Escolha antes de rodar; para
# trocar depois, reinicie o ambiente de execução.
BACKBONE = "EfficientNetB4"
os.environ["RETINOPATIA_BACKBONE"] = BACKBONE

if not os.path.exists(PASTA_REPOSITORIO):
    !git clone {URL_REPOSITORIO} {PASTA_REPOSITORIO}

!git -C {PASTA_REPOSITORIO} fetch --tags origin
!git -C {PASTA_REPOSITORIO} checkout --detach {REVISAO_CODIGO}
!git -C {PASTA_REPOSITORIO} log -1 --date=short --format="Código no commit %h (%cd): %s"

# Torna o pacote "retinopatia" importável neste notebook
caminho_src = os.path.join(PASTA_REPOSITORIO, "src")

if caminho_src not in sys.path:
    sys.path.insert(0, caminho_src)

# Recarrega os módulos automaticamente quando esta célula traz código
# novo, evitando ter que reiniciar o ambiente a cada edição no VS Code.
# O Colab usa Python 3.12+, que não tem mais o módulo "imp", mas a
# versão do IPython instalada nele ainda o importa no autoreload.
# Um substituto mínimo resolve; se mesmo assim falhar, o notebook
# segue sem autoreload (aí é preciso reiniciar após cada git pull).
import importlib
import types

try:
    import imp
except ModuleNotFoundError:
    sys.modules["imp"] = types.ModuleType("imp")
    sys.modules["imp"].reload = importlib.reload

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as erro:
    print("Autoreload indisponível:", erro)

print("Repositório pronto em:", PASTA_REPOSITORIO)

## Configurações Iniciais, Importações

In [ ]:
from retinopatia import (
    ambiente_colab,
    avaliacao,
    configuracao,
    conjunto_dados,
    diagnostico,
    pipeline_dados,
    pre_processamento,
    treinamento,
    utilitarios,
)

# Importada diretamente para não conflitar com a variável `modelo`
from retinopatia.modelo import construir_modelo

utilitarios.configurar_sementes(configuracao.SEED)

# Precisa vir antes da construção do modelo
utilitarios.configurar_precisao_mista()

## Conexão com o Drive e Preparação do Ambiente

In [ ]:
ambiente_colab.montar_drive()

# Se as imagens processadas desta versão já estão no Drive, restaura o
# cache e extrai só o CSV de rótulos (pula ~50 min de extração e
# pré-processamento)
cache_restaurado = ambiente_colab.restaurar_cache_imagens()

ambiente_colab.extrair_dataset(somente_rotulos=cache_restaurado)

## Carregamento do CSV e Divisão Estratificada (Treino, Validação e Teste)

In [ ]:
dados, df_treino, df_val, df_teste = conjunto_dados.preparar_dataset()

## Inspeção visual

In [ ]:
pre_processamento.pre_visualizar_recorte(dados)

## Pré-processamento Offline (Evitar 40 min por época)

In [ ]:
resultados_pre_processamento = pre_processamento.pre_processar_dataset(dados)

# Guarda as imagens processadas no Drive para as próximas sessões
ambiente_colab.salvar_cache_imagens()

## Imagens com máscara suspeita

In [ ]:
suspeitas = resultados_pre_processamento[
    resultados_pre_processamento["status"] == configuracao.STATUS_SUSPEITO
]["image"]

pre_processamento.pre_visualizar_recorte(dados, imagens=suspeitas)

## Imagens extras (test de 2015, só no treino)

Acrescenta ao treino as 53.576 imagens do "test" da competição de 2015, com os
rótulos publicados depois pelos organizadores. Validação e teste não mudam.
Na primeira vez extrai e processa as imagens (~1 h) e guarda o cache no Drive;
nas próximas, restaura o cache em poucos minutos.

In [ ]:
if configuracao.USAR_DADOS_EXTRA:
    df_extra = conjunto_dados.carregar_dados_extra(dados)

    ambiente_colab.preparar_imagens_extra(df_extra)

    df_treino = conjunto_dados.ampliar_treino(df_treino, df_extra)

## Pipelines de Dados

In [ ]:
treino_data, val_data, teste_data = pipeline_dados.criar_geradores(
    df_treino,
    df_val,
    df_teste
)

pipeline_dados.verificar_escala(treino_data, val_data)

# Se o pipeline levar perto de 1 s por lote, a GPU vai ficar esperando
pipeline_dados.medir_velocidade(treino_data)

## Cálculo dos Pesos das Classes

In [ ]:
pesos_dicionario = treinamento.calcular_pesos_classe(df_treino)

## Arquitetura do Modelo (Transfer Learning com EfficientNet)

In [ ]:
modelo, modelo_base = construir_modelo()

## Callbacks (Monitorização do Treino)

In [ ]:
# Registra no Drive os hiperparâmetros deste experimento
utilitarios.salvar_configuracao()

callbacks_fase1 = treinamento.criar_callbacks_fase1()

## Treino do Modelo (Fase 1 - Apenas as cabeças da rede)

In [ ]:
historico = treinamento.treinar_fase1(
    modelo,
    treino_data,
    val_data,
    pesos_dicionario,
    callbacks_fase1
)

## Avaliação, Gráficos e Matriz de Confusão (FASE 1)

In [ ]:
resultado_fase1 = avaliacao.avaliar_modelo(
    modelo,
    val_data,
    df_val,
    historico,
    "Fase 1"
)

## Fine-Tuning (Descongelando a Rede)

In [ ]:
callbacks_fase2 = treinamento.criar_callbacks_fase2()

historico_fase2 = treinamento.fine_tuning_fase2(
    modelo,
    modelo_base,
    treino_data,
    val_data,
    pesos_dicionario,
    callbacks_fase2
)

## Modelo completo

In [ ]:
# Salva o modelo completo (arquitetura + pesos) na pasta do experimento no Drive
modelo.save(configuracao.CAMINHO_MODELO_FINAL)

print("Modelo completo salvo em:", configuracao.CAMINHO_MODELO_FINAL)

## Avaliação, Gráficos e Matriz de Confusão (FASE 2)

In [ ]:
resultado_fase2 = avaliacao.avaliar_modelo(
    modelo,
    val_data,
    df_val,
    historico_fase2,
    "Fase 2"
)

In [ ]:
# Estimativa final: limiares escolhidos na validação, aplicados ao teste
resultado_teste = avaliacao.avaliar_no_teste(
    modelo,
    teste_data,
    df_teste,
    resultado_fase2
)

# Acrescenta este treino à tabela de todos os experimentos no Drive
avaliacao.registrar_experimento(resultado_fase2, resultado_teste)

## Diagnóstico do modelo treinado

Erros por grau, retinopatia referenciável, cabeça de grau, agregação dos olhos
(comparações feitas na validação) e intervalos de confiança do teste.

In [ ]:
diagnostico_final = diagnostico.relatorio_completo(
    resultado_fase2,
    resultado_teste
)